In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
# Use the 'Files' pane on the left to find your folder,
# right-click it, and select 'Copy path'
folder_path = '/content/drive/MyDrive/Colab Notebooks/Ph.D. code'
#folder_path = '/content/drive/MyDrive/Ph.D./Git repo for thesis'

if folder_path not in sys.path:
    sys.path.append(folder_path)

In [1]:
!nvidia-smi

Wed Jun 24 10:43:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
############  SANITY TEST (STAGE 0 + 1) ##############

from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget

# Setup
budget = ResourceBudget(max_t_count=2, max_depth=10, max_gates=10)
dag = CircuitDAG(num_qubits=2)
state = CircuitState(dag=dag, budget=budget)

# Apply gates
print(state.apply_gate(Gate(GateType.H, (0,))))
print(state.apply_gate(Gate(GateType.CNOT, (0, 1))))
print(state.apply_gate(Gate(GateType.T, (1,))))
print(state.apply_gate(Gate(GateType.T, (1,))))
print(state.apply_gate(Gate(GateType.T, (1,))))  # should fail

print(state)

True
True
True
True
False
CircuitState(gates=4, depth=4, T=2)
H(0,) -> CNOT(0, 1) -> T(1,) -> T(1,)


In [ ]:
############  SANITY TEST (STAGE 2 + 3) ##############
from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget

dag = CircuitDAG(2)
budget = ResourceBudget(10, 10, 10)

state = CircuitState(dag, budget)

state.apply_gate(Gate(GateType.H, (0,)))
state.apply_gate(Gate(GateType.CNOT, (0, 1)))
state.apply_gate(Gate(GateType.T, (1,)))
state.apply_gate(Gate(GateType.S, (0,)))

print("STATE:")
print(state)

print("\nPHASE POLY:")
print(state.phase_poly)

print("\nTABLEAU:")
print(state.tableau)

STATE:
CircuitState(gates=4, depth=4, T=1)
H(0,) -> CNOT(0, 1) -> T(1,) -> S(0,)

PHASE POLY:
1*[0b10] + 2*[0b1]

TABLEAU:
X=
[[0 0]
 [0 1]]
Z=
[[1 0]
 [0 0]]
phase=[0 0]


In [ ]:
############  SANITY TEST (STAGE 4) PART 1 ##############
from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget
from canonical.canonicalizer import Canonicalizer

budget = ResourceBudget(10, 10, 10)

# Circuit A: S²
s1 = CircuitState(CircuitDAG(1), budget)
s1.apply_gate(Gate(GateType.S, (0,)))
s1.apply_gate(Gate(GateType.S, (0,)))

# Circuit B: T⁴
s2 = CircuitState(CircuitDAG(1), budget)
for _ in range(4):
    s2.apply_gate(Gate(GateType.T, (0,)))

canon = Canonicalizer()

print("Identity Hash A:", canon.identity_hash(s1))
print("Identity Hash B:", canon.identity_hash(s2))

print("Resource Hash A:", canon.resource_hash(s1))
print("Resource Hash B:", canon.resource_hash(s2))

Identity Hash A: c16c7bea25e875e4f284ad68c27ea32b5ff1e03b066b9b7b55009f7a9c1cceb5
Identity Hash B: c16c7bea25e875e4f284ad68c27ea32b5ff1e03b066b9b7b55009f7a9c1cceb5
Resource Hash A: 8f100b4bbf28840a1527ee8643ebef7fb20af2db7718a5091ec77c6a3a4fe2df
Resource Hash B: 27a994207696654ca5b3054c79efb2bb5a8104096a56efd23eda89d1e00a1543


In [ ]:
############  SANITY TEST (STAGE 4) PART 2 ##############
s3 = CircuitState(CircuitDAG(1), budget)
s3.apply_gate(Gate(GateType.H, (0,)))

print("Identity Hash S²:", canon.identity_hash(s1))
print("Identity Hash H :", canon.identity_hash(s3))

Identity Hash S²: c16c7bea25e875e4f284ad68c27ea32b5ff1e03b066b9b7b55009f7a9c1cceb5
Identity Hash H : 6af22f1bc2d94295cb210c6a0734b0d7459c92909665da49d949785ecea55bf8


In [ ]:
############  SANITY TEST (STAGE 5) ##############
from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget
from rl.features import extract_features

budget = ResourceBudget(10, 10, 10)
state = CircuitState(CircuitDAG(2), budget)

state.apply_gate(Gate(GateType.H, (0,)))
state.apply_gate(Gate(GateType.CNOT, (0, 1)))
state.apply_gate(Gate(GateType.T, (1,)))

features = extract_features(state)

print("Feature vector:", features)
print("Shape:", features.shape)

Feature vector: [0.1        0.3        0.3        0.9        0.7        0.7
 0.25       0.125      0.125      0.5        0.33333334 0.3       ]
Shape: (12,)


In [ ]:
############  SANITY TEST (STAGE 6) PART 1 ##############
from search.frontier import Frontier
from search.node import SearchNode

from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget

budget = ResourceBudget(10, 10, 10)

frontier = Frontier()

s1 = CircuitState(CircuitDAG(1), budget)
s1.apply_gate(Gate(GateType.T, (0,)))

node1 = SearchNode(priority=1.0, state=s1)

print("Insert s1:", frontier.push(node1))

Insert s1: True


In [ ]:
############  SANITY TEST (STAGE 6) PART 2 ##############

budget = ResourceBudget(10, 10, 10)
frontier = Frontier()

# Circuit A: S² (cheaper)
s1 = CircuitState(CircuitDAG(1), budget)
s1.apply_gate(Gate(GateType.S, (0,)))
s1.apply_gate(Gate(GateType.S, (0,)))

node1 = SearchNode(priority=2.0, state=s1)
print("Insert s1:", frontier.push(node1))  # True


# Circuit B: T⁴ (more expensive, same identity)
s2 = CircuitState(CircuitDAG(1), budget)
for _ in range(4):
    s2.apply_gate(Gate(GateType.T, (0,)))

node2 = SearchNode(priority=4.0, state=s2)
print("Insert s2:", frontier.push(node2))  # Should be False

Insert s1: True
Insert s2: False


In [ ]:
############  SANITY TEST (STAGE 6) PART 3 ##############
frontier = Frontier()

# Insert worse first
s_big = CircuitState(CircuitDAG(1), budget)
for _ in range(3):
    s_big.apply_gate(Gate(GateType.T, (0,)))

node_big = SearchNode(priority=3.0, state=s_big)
print("Insert big:", frontier.push(node_big))

# Insert better later
s_small = CircuitState(CircuitDAG(1), budget)
s_small.apply_gate(Gate(GateType.T, (0,)))

node_small = SearchNode(priority=1.0, state=s_small)
print("Insert small:", frontier.push(node_small))

Insert big: True
Insert small: True


In [ ]:
############  SANITY TEST (STAGE 6) PART 4 ##############
s_h = CircuitState(CircuitDAG(1), budget)
s_h.apply_gate(Gate(GateType.H, (0,)))

node_h = SearchNode(priority=1.0, state=s_h)

print("Insert H:", frontier.push(node_h))

Insert H: True


In [ ]:
############  SANITY TEST (STAGE 7) PART 1 ##############
from search.action_space import generate_actions

actions = generate_actions(2)

print("Num actions:", len(actions))
print(actions[:10])

Num actions: 8
[H(0,), S(0,), T(0,), H(1,), S(1,), T(1,), CNOT(0, 1), CNOT(1, 0)]


In [ ]:
############  SANITY TEST (STAGE 7) PART 2 ##############
from search.expansion import expand_node
from search.node import SearchNode
from search.action_space import generate_actions

from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from ckt_types import ResourceBudget

budget = ResourceBudget(10, 10, 10)
state = CircuitState(CircuitDAG(2), budget)

root = SearchNode(priority=0.0, state=state)

actions = generate_actions(2)

children = expand_node(root, actions)

print("Num children:", len(children))
print(children[:5])

Num children: 8
[Node(priority=1.0000, state=CircuitState(gates=1, depth=1, T=0)
H(0,)), Node(priority=1.0000, state=CircuitState(gates=1, depth=1, T=0)
S(0,)), Node(priority=2.0000, state=CircuitState(gates=1, depth=1, T=1)
T(0,)), Node(priority=1.0000, state=CircuitState(gates=1, depth=1, T=0)
H(1,)), Node(priority=1.0000, state=CircuitState(gates=1, depth=1, T=0)
S(1,))]


In [ ]:
############  SANITY TEST (STAGE 7) PART 3 ##############
budget = ResourceBudget(max_t_count=1, max_depth=10, max_gates=10)

state = CircuitState(CircuitDAG(1), budget)

root = SearchNode(priority=0.0, state=state)

actions = generate_actions(1)

children = expand_node(root, actions)

# Now expand again from a T state
t_child = [c for c in children if c.state.t_count == 1][0]

children2 = expand_node(t_child, actions)

print("Children after T budget exhausted:")
for c in children2:
    print(c.state.t_count)

Children after T budget exhausted:
1
1


In [ ]:
############  SANITY TEST (STAGE 8) PART 1 ##############
from certification.algebraic import AlgebraicCertificationEngine
from certification.composite import CompositeCertificationEngine

from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from circuit.gate import Gate
from enums import GateType
from ckt_types import ResourceBudget

# Target = T^2
target_state = CircuitState(CircuitDAG(1), ResourceBudget(10,10,10))
target_state.apply_gate(Gate(GateType.T, (0,)))
target_state.apply_gate(Gate(GateType.T, (0,)))

target_terms = tuple(sorted(
    (m, c % 8) for m, c in target_state.phase_poly.terms.items()
))

engine = CompositeCertificationEngine([
    AlgebraicCertificationEngine(target_terms)
])

# Test state
test_state = CircuitState(CircuitDAG(1), ResourceBudget(10,10,10))
test_state.apply_gate(Gate(GateType.S, (0,)))  # S = T^2

result = engine.certify(test_state)

print(result.status, result.score)

CertStatus.SUCCESS 1.0


In [ ]:
############  SANITY TEST (STAGE 8) PART 2 ##############
bad_state = CircuitState(CircuitDAG(1), ResourceBudget(10,10,10))
bad_state.apply_gate(Gate(GateType.T, (0,)))

result = engine.certify(bad_state)

print(result.status)

CertStatus.FAILURE


In [ ]:
############  SANITY TEST (STAGE 9) PART 1 ##############
from rl.policy import LinearQPolicy
from rl.features import extract_features

from circuit.circuit_state import CircuitState
from circuit.dag import CircuitDAG
from ckt_types import ResourceBudget

policy = LinearQPolicy(feature_dim=12)

state = CircuitState(CircuitDAG(1), ResourceBudget(10,10,10))

print("Q value:", policy.q_value(state))

Q value: 0.0


In [ ]:
############  SANITY TEST (STAGE 9) PART 2 ##############
from search.action_space import generate_actions

actions = generate_actions(1)

result = policy.select_action(state, actions)

print(result)

(H(0,), 0.0, CircuitState(gates=1, depth=1, T=0)
H(0,))


In [ ]:
############  SANITY TEST (STAGE 9) PART 3 ##############
next_state = result[2]

policy.update(
    state=state,
    reward=1.0,
    next_state=next_state,
    done=False
)

print("Updated theta:", policy.theta)

Updated theta: [0.   0.   0.   0.01 0.01 0.01 0.   0.   0.   0.   0.   0.  ]


In [ ]:
############  SANITY TEST (STAGE 10) ##############
from env.rl_env import CircuitSynthesisEnv
from config import Config
from ckt_types import ResourceBudget

from certification.algebraic import AlgebraicCertificationEngine
from certification.composite import CompositeCertificationEngine

# target: T^2
target_terms = ((1, 2),)

cert_engine = CompositeCertificationEngine([
    AlgebraicCertificationEngine(target_terms)
])

config = Config(
    num_qubits=1,
    budget=ResourceBudget(10, 10, 10),
    max_steps=20
)

env = CircuitSynthesisEnv(config, cert_engine)

obs, _ = env.reset()

print("Initial obs:", obs)

done = False

while not done:
    action = env.action_space.sample()

    obs, reward, done, _, info = env.step(action)

    print("Reward:", reward, "Info:", info)

print("Finished")

Initial obs: [0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0.]
Reward: 10.0 Info: {'cert_status': 'SUCCESS', 'inserted': True}
Finished


In [ ]:
############  SANITY TEST (STAGE 11) ##############
from config import Config
from ckt_types import ResourceBudget

from env.rl_env import CircuitSynthesisEnv

from certification.algebraic import AlgebraicCertificationEngine
from certification.composite import CompositeCertificationEngine

from train import Trainer


# =========================================================
# Target: T^2
# =========================================================

target_terms = ((1, 2),)

cert_engine = CompositeCertificationEngine([
    AlgebraicCertificationEngine(target_terms)
])

config = Config(
    num_qubits=1,
    budget=ResourceBudget(10, 10, 10),
    max_steps=15
)

env = CircuitSynthesisEnv(config, cert_engine)

trainer = Trainer(env)

# Run very small training
trainer.train(num_episodes=10)

Episode 000 | Reward: -9.860 | Steps: 10 | Epsilon: 0.199
Episode 001 | Reward: -9.960 | Steps: 10 | Epsilon: 0.198
Episode 002 | Reward: -9.738 | Steps: 10 | Epsilon: 0.197
Episode 003 | Reward: -9.912 | Steps: 10 | Epsilon: 0.196
Episode 004 | Reward: -9.941 | Steps: 10 | Epsilon: 0.195
Episode 005 | Reward: -9.956 | Steps: 10 | Epsilon: 0.194
Episode 006 | Reward: -9.970 | Steps: 10 | Epsilon: 0.193
Episode 007 | Reward: -9.975 | Steps: 10 | Epsilon: 0.192
Episode 008 | Reward: -9.978 | Steps: 10 | Epsilon: 0.191
Episode 009 | Reward: 1.000 | Steps: 1 | Epsilon: 0.190
